In [24]:
import numpy as np

from numba import njit

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

import optuna
from optuna.visualization import plot_pareto_front
from optuna.trial import TrialState

from dysts.maps import Henon

import time

import pandas as pd
import json
import os

# Init

## Init Funcs

In [10]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scatter(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scatter(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [11]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [12]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [13]:
def weight_plot(weights, dims):
    dim_names = ["x", "y", "z"][:dims]
    num_features_per_type = len(weights) // 2
    num_nodes = num_features_per_type // dims

    labels = []
    for n in range(num_nodes):
        for d in range(dims):
            labels.append(f"Node {n + 1} {dim_names[d]} Pos")
            labels.append(f"Node {n + 1} {dim_names[d]} Vel")

    weights_flat = weights.flatten()
    pos_part = weights_flat[:num_features_per_type]
    vel_part = weights_flat[num_features_per_type:]

    combined_weights = np.empty_like(weights_flat)
    combined_weights[0::2] = pos_part
    combined_weights[1::2] = vel_part

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=combined_weights,
                marker_color=np.where(combined_weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Node State & Dimension",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [14]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    wall_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"
    if wall_nodes is not None and wall_nodes[0] != -1:
        node_colors[wall_nodes] = "blue"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

## Calc Init

In [15]:
@njit(cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))
    disp_reshaped = disp.reshape(num_nodes, dims)

    for i in range(len(connections_list)):
        idx_a = connections_list[i, 0]
        idx_b = connections_list[i, 1]

        delta = np.zeros(dims)
        dist_sq = 0.0
        for j in range(dims):
            pos_a = initial_pos[idx_a, j] + disp_reshaped[idx_a, j]
            pos_b = initial_pos[idx_b, j] + disp_reshaped[idx_b, j]
            delta[j] = pos_b - pos_a
            dist_sq += delta[j] ** 2

        dist = np.sqrt(dist_sq)

        mag = k_vals[i] * (dist - rest_lens[i])

        for j in range(dims):
            f_component = mag * (delta[j] / dist)
            forces[idx_a, j] += f_component
            forces[idx_b, j] -= f_component

    return forces.reshape(-1)

In [16]:
@njit(cache=True)
def run_simulation(
    steps, dt, m_inv_diag, c_diag, U, initial_pos, connections_list, k_vals, rest_lens, wall_nodes=[-1]
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2
        
        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + .5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Check Results

In [105]:
# This is for trial with one core at a time or n_jobs=1
trial_id = 485655
output_path = os.path.join("2_bayesian_mem_cap_order", "output", str(trial_id))
with open(os.path.join(output_path, "config.json"), "r") as f:
    config = json.load(f)
print("Config:", config)
df_ordered = pd.read_csv(os.path.join(output_path, "best_results.csv"))
print(df_ordered.shape)
df_ordered

Config: {'slurm_job_id': '485655', 'lattice': {'N': 30, 'dist_between': 2.5, 'dist_y': 1.0, 'wall_rest_multiplier': 1.1, 'target_node_count': 10}, 'simulation': {'dt': 0.01, 'total_steps': 9000, 'transient': 1000, 'train_steps': 5000, 'test_steps': 3000}, 'capacity': {'k_max': 200, 'k_cross': 25, 'k_cross3': 25}, 'optimization': {'num_eval_seeds': 5, 'fixed_seeds': [191664963, 1662057957, 1405681631, 942484272, 929893137]}}
(252, 10)


,trial,linear_mc,nonlinear_mc,total_mc,input_force,m_val,c_val,k_wall_val,k_between_val,ridge_alpha
0,2851,22.136705,34.197958,56.334663,6.288799,0.009570,0.489678,40.291153,78.913050,0.000001
1,4235,32.720682,11.502494,44.223176,1.010060,0.009570,0.144725,4.949274,78.913050,0.000001
2,4348,30.764207,21.952188,52.716395,1.805400,0.008698,0.184723,16.966954,78.913050,0.000002
3,4462,33.150725,3.438517,36.589243,1.010060,0.034633,0.144725,12.025559,78.913050,0.000001
4,4554,30.764207,21.952188,52.716395,1.805400,0.008698,0.184723,16.966954,78.913050,0.000002
...,...,...,...,...,...,...,...,...,...,...
247,8668,28.311181,27.111347,55.422528,3.429611,0.009570,0.333194,21.639103,80.973464,0.000001
248,8669,33.032020,7.067311,40.099331,1.010060,0.009570,0.064418,8.858211,78.913050,0.000001
249,8672,29.101300,26.072605,55.173905,3.429611,0.009570,0.291891,21.639103,80.973464,0.000001
250,8673,24.293738,32.430053,56.723791,4.014840,0.009570,0.419007,53.198319,78.913050,0.000001


In [106]:
with open(os.path.join(output_path, "config.json"), "r") as file:
    order_config = json.load(file)

In [81]:
top_nonlinear = df_ordered.sort_values(by="nonlinear_mc", ascending=False).iloc[:10]
top_linear = df_ordered.sort_values(by="linear_mc", ascending=False).iloc[:10]
top = df_ordered.sort_values(by="total_mc", ascending=False).iloc[:10]
all_data = [
    top,
    top_linear,
    top_nonlinear,
]


cols = 2
rows = (len(top) + cols - 1) // cols


for data_index, data in enumerate(all_data):
    subplot_titles = [
        f"Trial #{t.trial} | Total: {t.total_mc:.2f}" for idx, t in data.iterrows()
    ]

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=subplot_titles,
        vertical_spacing=0.08,
        horizontal_spacing=0.08,
    )

    i = 0
    for idx, trial in data.iterrows():
        r = (i // cols) + 1
        c = (i % cols) + 1
        values = trial.linear_mc, trial.nonlinear_mc

        fig.add_trace(
            go.Bar(
                x=["Linear", "Nonlinear"],
                y=values,
                text=[f"{v:.2f}" for v in values],
                textposition="auto",
                showlegend=False,
            ),
            row=r,
            col=c,
        )

        fig.update_xaxes(tickangle=-30, row=r, col=c)
        fig.update_yaxes(title_text="MC", row=r, col=c)
        i += 1

    fig.update_layout(
        title_text="top" if data_index == 0 else ("top_linear" if data_index == 1 else "top_nonlinear"),
        template="plotly_white",
        height=320 * rows,
        width=1100,
    )

    fig.show()

In [107]:
# This is for trial with one core at a time or n_jobs=1
trial_id = 485660
output_path = os.path.join("3_bayesian_mem_cap_disorder", "output", str(trial_id))
with open(os.path.join(output_path, "config.json"), "r") as f:
    config = json.load(f)
print("Config:", config)
df_disordered = pd.read_csv(os.path.join(output_path, "best_results.csv"))
print(df_disordered.shape)
df_disordered

Config: {'slurm_job_id': '485660', 'lattice': {'N': 30, 'dist_between': 2.5, 'dist_y': 1.0, 'target_node_count': 10}, 'simulation': {'dt': 0.01, 'total_steps': 9000, 'transient': 1000, 'train_steps': 5000, 'test_steps': 3000}, 'capacity': {'k_max': 200, 'k_cross': 25, 'k_cross3': 25}, 'optimization': {'num_eval_seeds': 5, 'fixed_seeds': [191664963, 1662057957, 1405681631, 942484272, 929893137]}}
(217, 16)


,trial,linear_mc,nonlinear_mc,total_mc,input_force,m_val,m_spread,c_val,c_spread,k_wall_val,k_wall_spread,k_between_val,k_between_spread,wall_rest_spread,between_rest_spread,ridge_alpha
0,3606,48.773979,4.784341,53.558319,5.245414,0.030380,0.007925,0.412058,0.718610,5.877472,1.080655,98.641671,0.608767,1.154027,0.915865,0.000002
1,3651,58.577566,1.253852,59.831417,5.245414,0.021967,0.089816,0.126151,0.871795,1.405608,1.289883,94.611460,0.332794,1.389064,0.801087,0.000004
2,3855,40.460233,13.332931,53.793163,5.245414,0.019125,0.081734,0.412058,0.871795,41.661548,0.729746,98.641671,0.146922,1.209110,0.801087,0.000002
3,4198,48.773979,4.784341,53.558319,5.245414,0.030380,0.007925,0.412058,0.718610,5.877472,1.080655,98.641671,0.608767,1.154027,0.915865,0.000002
4,4862,55.626648,1.658980,57.285628,5.245414,0.030380,0.081734,0.234537,0.871534,5.877472,1.484227,98.641671,0.608767,1.297491,0.915865,0.000002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,10949,20.846905,32.902747,53.749653,27.949869,0.024099,0.081734,0.807739,0.078617,41.661548,0.763846,85.697627,0.239545,0.951741,0.511150,0.000002
213,10986,38.586390,13.774861,52.361251,5.245414,0.024099,0.289883,0.412058,0.164976,19.154084,1.484227,98.641671,0.239545,0.813836,0.453741,0.000002
214,11005,19.979746,35.019465,54.999211,25.948462,0.018395,0.081734,0.807739,0.191898,41.661548,0.907409,98.641671,0.239545,0.198298,0.712342,0.000002
215,11012,45.901562,7.157869,53.059431,5.245414,0.023554,0.297566,0.312835,0.718610,19.154084,1.484227,94.611460,0.332794,1.154027,0.325229,0.000004


In [108]:
with open(os.path.join(output_path, "config.json"), "r") as file:
    disorder_config = json.load(file)

In [83]:
top_nonlinear = df_disordered.sort_values(by="nonlinear_mc", ascending=False).iloc[:10]
top_linear = df_disordered.sort_values(by="linear_mc", ascending=False).iloc[:10]
top = df_disordered.sort_values(by="total_mc", ascending=False).iloc[:10]
all_data = [
    top,
    top_linear,
    top_nonlinear,
]


cols = 2
rows = (len(top) + cols - 1) // cols


for data_index, data in enumerate(all_data):
    subplot_titles = [
        f"Trial #{t.trial} | Total: {t.total_mc:.2f}" for idx, t in data.iterrows()
    ]

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=subplot_titles,
        vertical_spacing=0.08,
        horizontal_spacing=0.08,
    )

    i = 0
    for idx, trial in data.iterrows():
        r = (i // cols) + 1
        c = (i % cols) + 1
        values = trial.linear_mc, trial.nonlinear_mc

        fig.add_trace(
            go.Bar(
                x=["Linear", "Nonlinear"],
                y=values,
                text=[f"{v:.2f}" for v in values],
                textposition="auto",
                showlegend=False,
            ),
            row=r,
            col=c,
        )

        fig.update_xaxes(tickangle=-30, row=r, col=c)
        fig.update_yaxes(title_text="MC", row=r, col=c)
        i += 1

    fig.update_layout(
        title_text=(
            "top"
            if data_index == 0
            else ("top_linear" if data_index == 1 else "top_nonlinear")
        ),
        template="plotly_white",
        height=320 * rows,
        width=1100,
    )

    fig.show()

# Run Trial

In [84]:
N = 30
dist_between = 2.5
dist_y = 1.0
x = np.zeros(N * 3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between * i
y = np.tile(np.array([0, 1 * dist_y, 2 * dist_y]), N)
nodes_pos = np.column_stack((x, y))

dt = 0.01
target_node_count = 10
mu = lambda x, sigma: np.log(x) - (sigma**2 / 2)

k_max = int(2 * 100)
transient = 1000
train_steps = 5000
test_steps = 3000
start_idx = max(transient, k_max)
total_steps = start_idx + train_steps + test_steps

N_step = int(N / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3
starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()
num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_ids = np.arange(x.size)
starts = 3 * np.arange(N)
connection_src = np.column_stack([starts, starts + 1]).flatten()
connection_dst = np.column_stack([starts + 1, starts + 2]).flatten()
betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]
src_nodes = np.concatenate([connection_src, between_src])
dst_nodes = np.concatenate([connection_dst, between_dst])
connections_list = np.column_stack((src_nodes, dst_nodes))

In [85]:
def evaluate_capacity(X_train, X_test, Y_train, Y_test, num_lags, ridge_alpha):
    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    c_k = np.zeros(num_lags)
    for k in range(num_lags):
        y_true = Y_test[:, k]
        y_hat = Y_pred[:, k]

        cov_matrix = np.cov(y_true, y_hat)
        cov = cov_matrix[0, 1]
        var_true = cov_matrix[0, 0]
        var_pred = cov_matrix[1, 1]

        denom = var_true * var_pred
        c_k[k] = (cov**2) / denom if denom > 1e-12 else 0.0

    return c_k

In [86]:
def spring_trial(
    rng_seed,
    input_force,
    m_val,
    m_spread,
    c_val,
    c_spread,
    k_wall_val,
    k_wall_spread,
    k_between_val,
    k_between_spread,
    wall_rest_val,
    wall_rest_spread,
    between_rest_val,
    between_rest_spread,
):
    rng = np.random.default_rng(rng_seed)

    u = rng.uniform(-1.0, 1.0, size=(total_steps, 1))
    U = np.zeros((total_steps, matrix_size))
    for j, node_index in enumerate(target_nodes):
        U[::, node_index * dims] = u.reshape(-1)
    U = U * input_force

    m_nodes = rng.lognormal(mean=mu(m_val, m_spread), sigma=m_spread, size=num_nodes)
    m_diag = np.repeat(m_nodes, dims)
    m_inv_diag = 1.0 / m_diag

    c_nodes = rng.lognormal(mean=mu(c_val, c_spread), sigma=c_spread, size=num_nodes)
    c_diag = np.repeat(c_nodes, dims)

    k_connection_vals = rng.lognormal(
        mean=mu(k_wall_val, k_wall_spread),
        sigma=k_wall_spread,
        size=connection_src.shape[0] // 2,
    ).repeat(2)
    k_between_vals = rng.lognormal(
        mean=mu(k_between_val, k_between_spread),
        sigma=k_between_spread,
        size=between_src.shape[0],
    )
    k_vals = np.concatenate([k_connection_vals, k_between_vals])

    rest_connection_lens = rng.lognormal(
        mean=mu(wall_rest_val, wall_rest_spread),
        sigma=wall_rest_spread,
        size=connection_src.shape[0] // 2,
    ).repeat(2)
    rest_between_lens = rng.lognormal(
        mean=mu(between_rest_val, between_rest_spread),
        sigma=between_rest_spread,
        size=between_src.shape[0],
    )
    rest_lens = np.concatenate([rest_connection_lens, rest_between_lens])

    displacement, velocity = run_simulation(
        steps=total_steps,
        dt=dt,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        rest_lens=rest_lens,
        wall_nodes=wall_nodes,
    )

    movement_nodes = 1 + np.arange(N) * 3
    movement_idx = movement_nodes * dims
    X = np.column_stack((displacement[:, movement_idx], velocity[:, movement_idx]))

    return X, u

In [87]:
k_cross = 25
k_cross3 = 25

def memory_trial(
    input, X_states, k_max, total_steps, start_idx, train_steps, test_steps, ridge_alpha
):
    u = input
    X = X_states

    X_clean = X[start_idx:]
    X_train = X_clean[:train_steps]
    X_test = X_clean[train_steps : train_steps + test_steps]

    Y_deg1 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg1[k:, k - 1] = np.sqrt(3.0) * u_delayed
    Y_deg2 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg2[k:, k - 1] = np.sqrt(5.0) * (3.0 * (u_delayed**2) - 1.0) / 2.0

    Y_deg3 = np.zeros((total_steps, k_max))
    for k in range(1, k_max + 1):
        u_delayed = u[:-k, 0]
        Y_deg3[k:, k - 1] = (
            np.sqrt(7.0) * (5.0 * (u_delayed**3) - 3.0 * u_delayed) / 2.0
        )

    # Degree 1
    Y1_clean = Y_deg1[start_idx:]
    Y1_train = Y1_clean[:train_steps]
    Y1_test = Y1_clean[train_steps : train_steps + test_steps]
    c_deg1 = evaluate_capacity(X_train, X_test, Y1_train, Y1_test, k_max, ridge_alpha)
    # Degree 2
    Y2_clean = Y_deg2[start_idx:]
    Y2_train = Y2_clean[:train_steps]
    Y2_test = Y2_clean[train_steps : train_steps + test_steps]
    c_deg2 = evaluate_capacity(X_train, X_test, Y2_train, Y2_test, k_max, ridge_alpha)
    # Degree 3
    Y3_clean = Y_deg3[start_idx:]
    Y3_train = Y3_clean[:train_steps]
    Y3_test = Y3_clean[train_steps : train_steps + test_steps]
    c_deg3 = evaluate_capacity(X_train, X_test, Y3_train, Y3_test, k_max, ridge_alpha)

    cross_pairs = [
        (k1, k2) for k1 in range(1, k_cross + 1) for k2 in range(k1 + 1, k_cross + 1)
    ]
    num_cross_pairs = len(cross_pairs)
    Y_cross2 = np.zeros((total_steps, num_cross_pairs))
    for idx, (k1, k2) in enumerate(cross_pairs):
        max_k = k2
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]
        Y_cross2[max_k:, idx] = 3.0 * (u_k1 * u_k2)

    # Cross Degree 2
    Y_cross2_clean = Y_cross2[start_idx:]
    c_cross2 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross2_clean[:train_steps],
        Y_cross2_clean[train_steps : train_steps + test_steps],
        num_cross_pairs,
        ridge_alpha,
    )

    # Cross Degree 3 (2+1 interaction)
    pairs_21 = [
        (k1, k2)
        for k1 in range(1, k_cross3 + 1)
        for k2 in range(1, k_cross3 + 1)
        if k1 != k2
    ]
    num_21 = len(pairs_21)
    Y_cross3_21 = np.zeros((total_steps, num_21))
    for idx, (k1, k2) in enumerate(pairs_21):
        max_k = max(k1, k2)
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]

        # sqrt(5) * P2 * sqrt(3) * P1 = sqrt(15) * ((3*u^2 - 1)/2) * u
        p2 = np.sqrt(5.0) * (3.0 * (u_k1**2) - 1.0) / 2.0
        p1 = np.sqrt(3.0) * u_k2
        Y_cross3_21[max_k:, idx] = p2 * p1
    # Cross Degree 3 (1+1+1 interaction)
    triples_111 = [
        (k1, k2, k3)
        for k1 in range(1, k_cross3 + 1)
        for k2 in range(k1 + 1, k_cross3 + 1)
        for k3 in range(k2 + 1, k_cross3 + 1)
    ]
    num_111 = len(triples_111)
    Y_cross3_111 = np.zeros((total_steps, num_111))
    for idx, (k1, k2, k3) in enumerate(triples_111):
        max_k = k3
        u_k1 = u[max_k - k1 : len(u) - k1, 0]
        u_k2 = u[max_k - k2 : len(u) - k2, 0]
        u_k3 = u[max_k - k3 : len(u) - k3, 0]

        Y_cross3_111[max_k:, idx] = (3.0 * np.sqrt(3.0)) * (u_k1 * u_k2 * u_k3)
    # Cross Degree 3 (2+1 interaction)
    c_cross3_21 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross3_21[start_idx : start_idx + train_steps],
        Y_cross3_21[start_idx + train_steps : start_idx + train_steps + test_steps],
        num_21,
        ridge_alpha,
    )
    # Cross Degree 3 (1+1+1 interaction)
    c_cross3_111 = evaluate_capacity(
        X_train,
        X_test,
        Y_cross3_111[start_idx : start_idx + train_steps],
        Y_cross3_111[start_idx + train_steps : start_idx + train_steps + test_steps],
        num_111,
        ridge_alpha,
    )

    return [
        c_deg1,
        c_deg2,
        c_deg3,
        c_cross2,
        c_cross3_21,
        c_cross3_111,
    ]

In [99]:
top_order_val = df_ordered.sort_values(by="total_mc", ascending=False).iloc[0]
top_order_val

trial            8673.000000
linear_mc          24.293738
nonlinear_mc       32.430053
total_mc           56.723791
input_force         4.014840
m_val               0.009570
c_val               0.419007
k_wall_val         53.198319
k_between_val      78.913050
ridge_alpha         0.000001
Name: 250, dtype: float64

In [118]:
rng_trials_results_avg = np.zeros(6)
wall_rest_multiplier = 1.1
trial_seeds = order_config["optimization"]["fixed_seeds"]
for trial_seed in trial_seeds:
    X, u = spring_trial(
        rng_seed=trial_seed,
        input_force=top_order_val.input_force,
        m_val=top_order_val.m_val,
        m_spread=0,
        c_val=top_order_val.c_val,
        c_spread=0,
        k_wall_val=top_order_val.k_wall_val,
        k_wall_spread=0,
        k_between_val=top_order_val.k_between_val,
        k_between_spread=0,
        wall_rest_val=dist_y * wall_rest_multiplier,
        wall_rest_spread=0,
        between_rest_val=dist_between,
        between_rest_spread=0,
    )
    if np.isnan(X).any():
        raise optuna.TrialPruned("NaN detected in spring_trial results.")
    values = memory_trial(
        u,
        X,
        k_max,
        total_steps,
        start_idx,
        train_steps,
        test_steps,
        ridge_alpha=top_order_val.ridge_alpha,
    )
    values_sum = np.array([np.sum(c) for c in values])
    rng_trials_results_avg += values_sum

results = rng_trials_results_avg / len(trial_seeds)
linear_cap = results[0]
nonlinear_cap = results[1] + results[2] + results[3] + results[4] + results[5]
linear_cap, nonlinear_cap

(np.float64(24.2937382078516), np.float64(32.43005288083775))

In [119]:
fig = go.Figure()

categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

subplot_titles = (
    f"Order Trial| Total: {top_order_val.total_mc:.2f}"
)

fig.add_trace(
    go.Bar(
        x=categories,
        y=results,
        text=[f"{v:.2f}" for v in results],
        textposition="auto",
        showlegend=False,
    ),
)
fig.update_layout(
    title_text="Top Optuna Trials - Memory Capacity Profiles (Sorted by Total MC)",
    template="plotly_white"
)

fig.show()

In [101]:
top_disorder_val = df_disordered.sort_values(by="nonlinear_mc", ascending=False).iloc[0]
top_disorder_val

trial                  10540.000000
linear_mc                 18.104931
nonlinear_mc              35.043873
total_mc                  53.148803
input_force               44.602236
m_val                      0.018395
m_spread                   0.081734
c_val                      0.807739
c_spread                   0.191898
k_wall_val                41.661548
k_wall_spread              0.907409
k_between_val             98.641671
k_between_spread           0.239545
wall_rest_spread           0.198298
between_rest_spread        0.712342
ridge_alpha                0.000004
Name: 179, dtype: float64

In [116]:
rng_trials_results_avg = np.zeros(6)
wall_rest_multiplier = 1.1
trial_seeds = disorder_config["optimization"]["fixed_seeds"]
for trial_seed in trial_seeds:
    X, u = spring_trial(
        rng_seed=trial_seed,
        input_force=top_disorder_val.input_force,
        m_val=top_disorder_val.m_val,
        m_spread=top_disorder_val.m_spread,
        c_val=top_disorder_val.c_val,
        c_spread=top_disorder_val.c_spread,
        k_wall_val=top_disorder_val.k_wall_val,
        k_wall_spread=top_disorder_val.k_wall_spread,
        k_between_val=top_disorder_val.k_between_val,
        k_between_spread=top_disorder_val.k_between_spread,
        wall_rest_val=dist_y,
        wall_rest_spread=top_disorder_val.wall_rest_spread,
        between_rest_val=dist_between,
        between_rest_spread=top_disorder_val.between_rest_spread,
    )
    if np.isnan(X).any():
        raise optuna.TrialPruned("NaN detected in spring_trial results.")
    values = memory_trial(
        u,
        X,
        k_max,
        total_steps,
        start_idx,
        train_steps,
        test_steps,
        ridge_alpha=top_order_val.ridge_alpha,
    )
    values_sum = np.array([np.sum(c) for c in values])
    rng_trials_results_avg += values_sum

results = rng_trials_results_avg / len(trial_seeds)
linear_cap = results[0]
nonlinear_cap = results[1] + results[2] + results[3] + results[4] + results[5]
linear_cap, nonlinear_cap

(np.float64(18.10501142321003), np.float64(35.044846054168126))

In [117]:
fig = go.Figure()

categories = [
    "Deg 1 Linear",
    "Deg 2 Single",
    "Deg 3 Single",
    "Deg 2 Cross (1+1)",
    "Deg 3 Cross (2+1)",
    "Deg 3 Cross (1+1+1)",
]

subplot_titles = f"Order Trial| Total: {top_order_val.total_mc:.2f}"

fig.add_trace(
    go.Bar(
        x=categories,
        y=results,
        text=[f"{v:.2f}" for v in results],
        textposition="auto",
        showlegend=False,
    ),
)
fig.update_layout(
    title_text="Top Optuna Trials - Memory Capacity Profiles (Sorted by Total MC)",
    template="plotly_white",
)

fig.show()

# Pareto Front

In [127]:
# This is for trial with one core at a time or n_jobs=1
trial_id = 485655
output_path = os.path.join("2_bayesian_mem_cap_order", "output", str(trial_id))
df_ordered = pd.read_csv(os.path.join(output_path, "trials_results.csv"))
completed_df_ordered = df_ordered[df_ordered["state"] != "PRUNED"]
completed_df_ordered.shape

(7996, 15)

In [128]:
import plotly.express as px

# 1. Interactive scatter plot with full trial details on hover
fig = px.scatter(
    completed_df_ordered,
    x="values_0",
    y="values_1",
    hover_data=completed_df_ordered.columns,  # Displays all hyperparameters and metadata on hover
    title=f"Trial {trial_id}: Linear vs NonLine",
    template="plotly_white",
)

fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=1, color="black")))

fig.show()

In [129]:
# This is for trial with one core at a time or n_jobs=1
trial_id = 485660
output_path = os.path.join("3_bayesian_mem_cap_disorder", "output", str(trial_id))
df_disordered = pd.read_csv(os.path.join(output_path, "trials_results.csv"))
completed_df_disordered = df_disordered[df_disordered["state"] != "PRUNED"]
completed_df_disordered = df_disordered[df_disordered["state"] != "PRUNED"]
completed_df_disordered.shape

(7718, 21)

In [131]:
import plotly.express as px

# 1. Interactive scatter plot with full trial details on hover
fig = px.scatter(
    completed_df_disordered,
    x="values_0",
    y="values_1",
    hover_data=completed_df_disordered.columns,  # Displays all hyperparameters and metadata on hover
    title=f"Trial {trial_id}: Linear vs NonLine",
    template="plotly_white",
)

fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=1, color="black")))

fig.show()